# **Predicting Eye Color from Genes with Machine Learning**
IR: PNNL-SA-221036

****
**Step 1: Import Python Libraries**
****

Import statements are used to include code from one module to another. Here we are taking advantage of prewritten code and packages. One helpful example in machine learning is the scikit-learn (sklearn) library which supports different types of machine learning models, some of which we will use today.

In [ ]:
# import .. -> imports the whole module
import math

# import .. as .. -> imports the whole module and gives it an easy name to type
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
import itertools

# from .. import .. -> imports a specific part of a module
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from IPython.display import display, clear_output

****
**Step 2: Load Data**
****

To load the data, we are using pandas. Pandas is a python library that provides a data structure to work efficiently with structured data like comma-separated values (CSV) files (helping with data transformations like reshaping, data analysis like statistical operations, etc...).

*Note - you must upload final_no-nan.csv into google colab before loading the data. This is because google colab does not have access to your local machine's memory.*

<br>

**Dataset:**

ChrisTho23. (2024). Gen2PhenMini [Data set]. GitHub. https://github.com/ChrisTho23/Gen2PhenMini

In [ ]:
# Loading data using Pandas
df = pd.read_csv("final_no_nan.csv")

# Renaming data columns
names = ['eye_color', 'location_0', 'location_1', 'location_2', 'location_3', 'location_4', 'location_5']
df.columns = names

# Printing data table to show structure
print(df)

****
**Step 3: Split Data**
****

Once that data has been loaded we need to separate the features (data used to make the prediction) from the labels (correct outputs). We also need to split the data into train, validation, and test sets.

*Train set* -> used to train the model on predicting eye color

*Validation set* -> used to tune (change different parts of how the model learns) the model for best performance

*Test set* -> Used to approximate how good the final model is at predicting eye colors **(only used at the very end!)**

In [ ]:
# Separating SNP data (features or X) from eye_color (labels or Y)
feature_cols = ['location_0', 'location_1', 'location_2', 'location_3', 'location_4', 'location_5'] # list of all feature names
target_col = 'eye_color'                                                                            # String of label name
X = df[feature_cols]                                                                                # 2D dataset holding all features
Y = df[target_col]                                                                                  # 1D list dataset holding all labels

# Splitting data into 80% train, 10% validation, and 10% test
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
X_val, X_test, Y_val, Y_test = train_test_split(X_test, Y_test, test_size=0.5, random_state=42)

****
**Step 4: Visualize and Understand the Data**
****

This step is the vaguest as it will change depending on what your data is and what you want to do. Here we strive to understand how the data is separated between classes.

*Note - we are only visualizing training data. This is to reduce bias that will affect result clams later. (If we have seen the training dataset then we consciously or unconsciously change what we do to improve those results)*

In [ ]:
# Dictionary to map eye color number to eye color
eye_color_mapping = {0:'Blue', 1:'Blue-Green', 2:'Blue-Grey', 3:'Brown', 4:'Brown-Green', 5:'Green', 6:'Hazel'}

# Prints total training datapoints (how many different people in training set) and total number of classes (number of unique eye colors)
print("Number of training data points:", len(Y_train))
print("Number of classes:", Y_train.nunique())

print() # Prints a new line

# Prints a table of eye colors (name and values) as well as the number of people in the training dataset who have that eye color
counts = Y_train.value_counts()
print("Counts of each class:")
print(f"{'Eye Color String':<20}{'Eye Color Value':<20}{'Counts':<20}") # Printing table header
for i in range(len(counts)):
  print(f"{eye_color_mapping[i]:<20}{i:<20}{counts[i]:<20}") # Printing each row

print()

# Prints the max class and what percentage of the training dataset it represents
print(f"majority training class is {counts.idxmax()} by {((counts.max() / len(Y_train)) * 100):.2f}%")

****
**Code to Generate Figures**
****
Feel free to run the code and not worry about understanding it. Comments have been created if you do want to learn more at a later time.

Generates Heatmaps...




In [ ]:
# @title
### We will not focus too much on this code, but the heatmaps can be used to understand what genes are more important for predicting eye color. ###
fig, axes = plt.subplots(2, 3, figsize=(12, 8))                                                   # defines the number of heatmaps, the layout, and the size of each heatmap
axes = axes.flatten()

for i, col in enumerate(feature_cols):                                                            # loops through and generates each heatmap
    pivot = pd.crosstab(X_train[col], Y_train, dropna=False)                                      # dataset of the current gene and the associated eye color

    data = pivot.values.astype(float)                                                             # changes data values to type float

    im = axes[i].imshow(data, cmap='Blues', aspect='auto')                                        # creates the heatmap

    axes[i].set_title(col)                                                                        # titles the heatmap with the gene name
    axes[i].set_xticks(np.arange(pivot.shape[1]))                                                 # sets the number of eye color possible values (in this case 7)
    axes[i].set_xticklabels(pivot.columns, rotation=45, ha='right')                               # labels each eye color value
    axes[i].set_yticks(np.arange(pivot.shape[0]))                                                 # sets the number of gene expressions (in this case True or False)
    axes[i].set_yticklabels(pivot.index)                                                          # labels True or False

    vmax = data.max() if data.size > 0 else 1                                                     # finds the max sub heatmap cell value -> used for text color later
    for r in range(pivot.shape[0]):                                                               # loops through each row of the heatmap
        for c in range(pivot.shape[1]):                                                           # loops through search heatmap cell
            val = int(data[r, c])                                                                 # gets the current heatmap cell value
            text_color = 'white' if data[r, c] > 0.6 * vmax else 'black'                          # Sets the text color to black or white depending on how darkness of the heatmap cell
            axes[i].text(c, r, f'{val}', ha='center', va='center', color=text_color, fontsize=9)  # writes the value in each heatmap cell

    fig.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)                                        # adds the color bar to each heatmap

plt.tight_layout()                                                                                # helps layout -> improving readability to tick labels
plt.show()                                                                                        # shows the final plot below


Generates Two Label Scatter Plots...

In [ ]:
# @title
train_df = X_train.copy()                                               # Copy training features (SNP data) so that we do not edit our data
train_df[target_col] = Y_train.values                                   # Combine train features and labels into one DataFrame for plotting
train_df[feature_cols] = train_df[feature_cols].astype(int)             # Change binary features to numeric (0/1) instead of True/False values

def add_jitter(arr, scale=0.08):                                        # Small helper: add tiny random noise so points spread a bit
    return arr + np.random.normal(0, scale, size=len(arr))

classes = train_df[target_col].unique()                                 # Getting the values for each class
selected_classes = list(classes)[:2]
palette = plt.cm.get_cmap('tab10', len(selected_classes))               # Pick consistent colors per class
train_df = train_df[train_df[target_col].isin(selected_classes)].copy()

pairs = list(itertools.combinations(feature_cols, 2))                   # Creating all the feature pairs for each plot
ncols = 3                                                               # how many plots across
nrows = int(np.ceil(len(pairs) / ncols))                                # rows needed

fig, axes = plt.subplots(nrows, ncols, figsize=(4.5*ncols, 3.8*nrows))  # defines the number of plots, the layout, and the size of each plot
axes = np.ravel(axes)                                                   # flatten for easy looping

for i, (xf, yf) in enumerate(pairs):                                    # Loops through each pair to make each plot
    ax = axes[i]

    x = add_jitter(train_df[xf].values, scale=0.08)                     # Prepare jittered coordinates (still centered at 0 or 1) for x axis values
    y = add_jitter(train_df[yf].values, scale=0.08)                     # Prepare jittered coordinates (still centered at 0 or 1) for y axis values

    for ci, cls in enumerate(selected_classes):                         # Plot each class in a different color
        mask = (train_df[target_col] == cls).values                     # Creating a mask for all points in a different class than cls
        ax.scatter(x[mask], y[mask], s=40, alpha=0.85,                  # Plot
                   color=palette(ci), edgecolors='none')

    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['False', 'True'])                               # Make axes intuitive: ticks at 0 and 1 with labels False/True
    ax.set_yticklabels(['False', 'True'])
    ax.set_xlim(-0.35, 1.35); ax.set_ylim(-0.35, 1.35)
    ax.grid(True, linestyle='--', alpha=0.4)

    ax.set_title(f"{xf} vs {yf}", fontsize=10)                          # Adding title to each table
    ax.set_xlabel(xf); ax.set_ylabel(yf)                                # Adding labels to each axis

for j in range(len(pairs), len(axes)):                                  # hide any empty subplot slots
    axes[j].set_visible(False)

legend_handles = [plt.scatter([], [], color=palette(i), label=str(cls)) for i, cls in enumerate(selected_classes)]
fig.legend(handles=legend_handles, title=target_col,                    # one simple legend for the whole figure (dummy handles)
           loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0)

plt.tight_layout()
plt.subplots_adjust(right=0.85)                                         # make room for legend


Generates All Label Scatter Plots...

In [ ]:
# @title
train_df = X_train.copy()                                               # Copy training features (SNP data) so that we do not edit our data
train_df[target_col] = Y_train.values                                   # Combine train features and labels into one DataFrame for plotting
train_df[feature_cols] = train_df[feature_cols].astype(int)             # Change binary features to numeric (0/1) instead of True/False values

def add_jitter(arr, scale=0.08):                                        # Small helper: add tiny random noise so points spread a bit
    return arr + np.random.normal(0, scale, size=len(arr))

classes = train_df[target_col].unique()                                 # Getting the values for each class
palette = plt.cm.get_cmap('tab10', len(classes))                        # Pick consistent colors per class

pairs = list(itertools.combinations(feature_cols, 2))                   # Creating all the feature pairs for each plot
ncols = 3                                                               # how many plots across
nrows = int(np.ceil(len(pairs) / ncols))                                # rows needed

fig, axes = plt.subplots(nrows, ncols, figsize=(4.5*ncols, 3.8*nrows))  # defines the number of plots, the layout, and the size of each plot
axes = np.ravel(axes)                                                   # flatten for easy looping

for i, (xf, yf) in enumerate(pairs):                                    # Loops through each pair to make each plot
    ax = axes[i]

    x = add_jitter(train_df[xf].values, scale=0.08)                     # Prepare jittered coordinates (still centered at 0 or 1) for x axis values
    y = add_jitter(train_df[yf].values, scale=0.08)                     # Prepare jittered coordinates (still centered at 0 or 1) for y axis values

    for ci, cls in enumerate(classes):                                  # Plot each class in a different color
        mask = (train_df[target_col] == cls).values                     # Creating a mask for all points in a different class than cls
        ax.scatter(x[mask], y[mask], s=40, alpha=0.85,                  # Plot
                   color=palette(ci), edgecolors='none')

    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['False', 'True'])                               # Make axes intuitive: ticks at 0 and 1 with labels False/True
    ax.set_yticklabels(['False', 'True'])
    ax.set_xlim(-0.35, 1.35); ax.set_ylim(-0.35, 1.35)
    ax.grid(True, linestyle='--', alpha=0.4)

    ax.set_title(f"{xf} vs {yf}", fontsize=10)                          # Adding title to each table
    ax.set_xlabel(xf); ax.set_ylabel(yf)                                # Adding labels to each axis

for j in range(len(pairs), len(axes)):                                  # hide any empty subplot slots
    axes[j].set_visible(False)

legend_handles = [plt.scatter([], [], color=palette(i), label=str(cls)) for i, cls in enumerate(classes)]
fig.legend(handles=legend_handles, title=target_col,                    # one simple legend for the whole figure (dummy handles)
           loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0)

plt.tight_layout()
plt.subplots_adjust(right=0.85)                                         # make room for legend


****
**Step 5: Train Linear Classifier to Predict Eye Color**
****


Here we are implementing a Linear Classifier model called Logistic Regression. Below is the equation used to calculate predictions **(You do not have to look at this, but if you are curious I have provided it)**.

$$p = {softmax}(\sum_{i=1}^{n} w_ix_i + b)$$

where

$softmax(p_i) = \frac{e^{p_i}}{\sum_{j=1}^K e^{p_j}}$

$K = $ number of classes (in this case 7)

$n = $ number of data points (in this case 691 for training)

$w = $ learnable weights

$b = $ learnable biases


In [ ]:
logReg = LogisticRegression(class_weight='balanced')                # Define a linear classifier
logReg.fit(X_train, Y_train)                                        # Train the model on the training dataset (where X is SNP data and Y is eye color)
Y_pred = logReg.predict(X_val)                                      # Give the model only validation set SNP data and predict eye colors
accuracy = accuracy_score(Y_val, Y_pred)*100                        # Calculate what % the model got correct
print(f"Logistic Regression Validation Accuracy: {accuracy:.3f}%")  # print the accuracy

****
**Step 6: Improve the Base Linear Classifier**
****

Below are different factors that affect how our model will learn. These are pre-set during training and require the model to be retrained each time a value is changed.

**C** &rarr; tells the model how strictly it should try to get every training example right.
  * If C is big, then you are very strict. This can lead to memorizing training data and not learning the true pattern of the data.
  * If C is small, then you are more relaxed about getting things wrong. This can lead to not trying hard enough to learn how to predict eye color.
  * You want to find a happy middle where the model cares about learning how to solve the problem.

**Penalty** &rarr; Stops the model from getting too wild or overthinking too much.
  * None = No rules. Do whatever you want.
  * L2 = Look at everything, but don't look at any one thing too much.
  * L1 = Don't look at everything only look at the important things.
  * Elastic Net = Do a balance between L1 and L2.

**tolerance** &rarr; How patient we should be before saying "good enough".
  * Small tolerance = Be a perfectionist. Don’t quit early.
  * Large tolerance = Good enough is good enough.

**class_weight** &rarr; tells the model how much attention it should give to each class (each eye color) when learning.
  * None = Every class counts the same.
  * Balanced = Give rare classes extra volume so they’re heard.

**solver** &rarr; The method the computer uses to figure out the best line or boundary. (*note - not all penalties work for all solvers!*)
  * lbfgs = calm and steady problem solver. (must use L2 or None penalty)
  * sag/saga = great for big datasets; saga is the most flexible. (must use L2 or None penalty for sag. Must use L1, L2, Elastic Net, or None penalty for saga)
  * newton cg/newton-cholesky = intelligent big step solver. ( must use L2 or None penalty)

**max iteration** &rarr; A limit on how long the model is allowed to keep trying to learn.
  * Too few iterations = turning something in before it is finished.
  * Too many iterations = won't make the model smarter it will just take longer.

**warm start** &rarr; Tells the model whether to start learning from scratch or continue learning from where it left off.



In [ ]:
# @title
C = widgets.FloatLogSlider(value=1.0, base=10, min=-2, max=2, step=0.1, description='C (LogReg)')
penalty = widgets.Dropdown(options=[None, 'l1', 'l2', 'elasticnet'], value=None, description='Penalty')
tol = widgets.FloatLogSlider(value=1e-4, min=-10, max=0, step=1e-6, description='Tolerance')
class_weight = widgets.Dropdown(options=[None, 'balanced'], value=None, description='Class Weight')
solver = widgets.Dropdown(options=['newton-cg', 'lbfgs', 'sag', 'saga', 'newton-cholesky'], value='lbfgs', description='Solver')
max_iter = widgets.IntSlider(value=100, min=10, max=1000, step=10, description='Max Iterations')
warm_start = widgets.Checkbox(value=False, description='Warm Start')
save_model = widgets.Checkbox(value=False, description='Save Model?')
run_btn = widgets.Button(description="Train & Evaluate", button_style='success')

logReg_out = widgets.Output()

def run_logReg_experiment(_):
    global tuned_logReg
    with logReg_out:
        clear_output()
        tuning_logReg = LogisticRegression(C=C.value, penalty=penalty.value, tol=tol.value, class_weight=class_weight.value, solver=solver.value, max_iter=max_iter.value, warm_start=warm_start.value)
        tuning_logReg.fit(X_train, Y_train)
        lr_acc = tuning_logReg.score(X_val, Y_val)
        print(f"Logistic Regression Accuracy: {lr_acc:.3f}")
        if save_model.value:
            tuned_logReg = tuning_logReg

tuned_logReg = None
display(widgets.VBox([C, penalty, tol, class_weight, solver, max_iter, warm_start, save_model, run_btn, logReg_out]))
run_btn.on_click(run_logReg_experiment)



***
**Step 7: Test the Best Logistic Regression Models**
***

In [ ]:
'''
logReg_test_acc = logReg.score(X_test, Y_test)*100
print(f"Logistic Regression Test Accuracy: {logReg_test_acc:.3f}%")

if tuned_logReg is not None:
  tuned_logReg_test_acc = tuned_logReg.score(X_test, Y_test)*100
  print(f"Tuned Logistic Regression Test Accuracy: {tuned_logReg_test_acc:.3f}%\n")
else:
  print("Tuned logistic regression model was not saved. Please click \"save model?\" and then re-run \"Train & Evaluate")
'''

****
**Step 8: Train a Non-Linear Model**
****
In this next section we will try to improve apon the linear logistic regression model by using neural networks. Neural networks have the abilty to learn more complex functions to seperate classes beyond just linear (stright) lines.

Generates Training Loss and Validation Accuracy Graphs for Neural Networks...

In [ ]:
# @title
def create_graphs(model):
  if not hasattr(model, "loss_curve_"):
    print("No loss curve available. Use solver='sgd' or 'adam' to generate graph.")
  else:
    plt.figure(figsize=(6, 4))
    plt.plot(model.loss_curve_, label="Training Loss", color='blue')
    plt.title("Training Loss per Iteration")
    plt.xlabel("Iteration")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.legend()
    plt.show()

  if not hasattr(model, "validation_scores_") or model.validation_scores_ is None:
    print("No validation scores available. Use Early Stopping to generate graph.")
  else:
    plt.figure(figsize=(6, 4))
    plt.plot(model.validation_scores_, label="Validation Accuracy", color='green')
    plt.title("Validation Accuracy per Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1)
    plt.grid(True)
    plt.legend()
    plt.show()

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(50, 30),
    activation='relu',
    solver='adam',
    max_iter=1000,
    random_state=42,
    early_stopping=True
)                                                           # Define a neural network

mlp.fit(X_train, Y_train)                                   # Train the model on the training dataset (where X is SNP data and Y is eye color)
mlp_Y_pred = mlp.predict(X_val)                             # Give the model only validation set SNP data and predict eye colors
accuracy = accuracy_score(Y_val, mlp_Y_pred)*100            # Calculate what % the model got correct
print(f"MLP Accuracy: {accuracy:.3f}%")
create_graphs(mlp)

****
**Step 8: Tune Neural Network Hyperparameters**
****

**Hidden 1** -> Defines the number of neurons if the first hidden layer. The more neurons the more capacity to learn new patterns.

**Hidden 2** -> Almost the same as Hidden 1, but it defines the number of neurons in the second hidden layer.

**Activation** -> This is where the model gets non-linearity. Decides how much to weight different neurons. relu is fast and popular for deep learning. tanh is good for smaller networks. logistic is a sigmoid curve. identity is no non-linearity.

**Solver** -> Algorithms to adjust weights during training. sgd stands for stochastic gradient descent, this is the classic method. adam is an adaptive method (best default). lbfgs is good for smaller datasets.

**Alpha** -> Defines how "fancy" or complicated the model can be. Higher alpha means the model will be less complicated. This is used to make sure the model is not overfitting to the training dataset.

**Learning Rate** -> What method is used to adjust the weights (or change the prediction based on how bad the model did last time). constant used the same step size each time, adaptive slows down if the changes are not making the model better, invscaling will slowly decrease the learning rate over time.

**Learning Rate Init** -> Starting step size for weight updates. Big steps will help you learn faster but you risk overshooting the target. Smaller steps will have slower training, but safer for getting to the endpoint you want (assuming max iterations are big enough).

**Max Iterations** -> Maximum chances for the model to learn from the data before being tested. Too small and the model might not learn enough. Too high and the model might have memorized  the training data and preform worse on validation and test.

**Random State** -> Sets the randomness for the initial weights. The same seed is like assuring a deck of cards will be shuffled the same every time. (Note - the numbers don't correlate like a scale. Meaning higher seeds do NOT equal more random. To find the best seed you might just have to play around.)

**tolerance** -> If model improvement is smaller than this number, stop training. Think of this as how patient we should be before saying "good enough".

**warm start** -> If true, reuse the solution of the previous call to fit as initialization, otherwise, just erase the previous solution.

**Momentum (for SGD only)** -> If the weights are changing in the same way each time, then momentum will change them faster.

**Early Stopping** -> Stops training if validation score does not improve or gets worse.



In [ ]:
# @title
hidden1 = widgets.IntSlider(value=50, min=4, max=1024, step=5, description='Hidden1')
hidden2 = widgets.IntSlider(value=50, min=0, max=1024, step=5, description='Hidden2')
activation = widgets.Dropdown(options=['identity', 'logistic', 'tanh', 'relu'], value='identity', description='Activation')
solver = widgets.Dropdown(options=['lbfgs', 'sgd', 'adam'], value='sgd', description='Solver')
alpha = widgets.FloatLogSlider(value=1e-5, min=-10, max=0, step=1e-6, description='Alpha')
learning_rate = widgets.Dropdown(options=['constant', 'invscaling', 'adaptive'], value='constant', description='Learning Rate')
learning_rate_init = widgets.FloatLogSlider(value=0.001, min=-10, max=0, step=1e-6, description='Learning Rate Init')
max_iter = widgets.IntSlider(value=100, min=10, max=100000, step=10, description='Max Iterations')
random_state = widgets.IntSlider(value=42, min=0, max=100, step=1, description='Random State')
tol = widgets.FloatLogSlider(value=1e-4, min=-10, max=0, step=1e-6, description='Tolerance')
warm_start = widgets.Checkbox(value=False, description='Warm Start')
momentum = widgets.FloatSlider(value=0.9, min=0, max=1, step=0.01, description='Momentum (SGD)')
early_stopping = widgets.Checkbox(value=False, description='Early Stopping')
save_model = widgets.Checkbox(value=False, description='Save Model?')
run_btn = widgets.Button(description="Train & Evaluate", button_style='success')
test_btn = widgets.Button(description="Test", button_style='danger')

MLP_out = widgets.Output()

def run_MLP_experiment(_):
    global tuned_mlp
    with MLP_out:
        clear_output()
        layers = (hidden1.value,) if hidden2.value == 0 else (hidden1.value, hidden2.value)
        tuning_mlp = MLPClassifier(hidden_layer_sizes=layers, activation=activation.value, alpha=alpha.value, learning_rate_init=learning_rate_init.value, solver=solver.value, learning_rate=learning_rate.value, max_iter=max_iter.value, random_state=random_state.value, tol=tol.value, warm_start=warm_start.value, momentum=momentum.value, early_stopping=early_stopping.value)
        tuning_mlp.fit(X_train, Y_train)
        mlp_val_acc = tuning_mlp.score(X_val, Y_val)*100
        create_graphs(tuning_mlp)
        print(f"MLP Validation Accuracy: {mlp_val_acc:.3f}")
        if save_model.value:
          tuned_mlp = tuning_mlp

tuned_mlp = None
display(widgets.VBox([hidden1, hidden2, activation, solver, alpha, learning_rate, learning_rate_init, max_iter, random_state, tol, warm_start, momentum, early_stopping, save_model, run_btn, MLP_out]))
run_btn.on_click(run_MLP_experiment)

***
**Step 9: Test the Best Logistic Regression and Neural Network Models**
***

In [ ]:
'''
logReg_test_acc = logReg.score(X_test, Y_test)*100
print(f"Logistic Regression Test Accuracy: {logReg_test_acc:.3f}%")

if tuned_logReg is not None:
  tuned_logReg_test_acc = tuned_logReg.score(X_test, Y_test)*100
  print(f"Tuned Logistic Regression Test Accuracy: {tuned_logReg_test_acc:.3f}%\n")
else:
  print("Tuned logistic regression model was not saved. Please click \"save model?\" and then re-run \"Train & Evaluate")

mlp_test_acc = mlp.score(X_test, Y_test)*100
print(f"MLP Test Accuracy: {mlp_test_acc:.3f}%")

if tuned_mlp is not None:
  tuned_mlp_test_acc = tuned_mlp.score(X_test, Y_test)*100
  print(f"Tuned MLP Test Accuracy: {tuned_mlp_test_acc:.3f}%")
else:
  print("Tuned neural network was not saved. Please click \"save model?\" and then re-run \"Train & Evaluate")
'''